# 03 — ResNet50 Transfer Learning

Fine-tune a pretrained ResNet50 (ImageNet) for cattle breed classification.

### Two-Phase Training
1. **Phase 1** (10 epochs): Freeze backbone, train only the classifier head
2. **Phase 2** (20 epochs): Unfreeze `layer3` + `layer4`, fine-tune with lower LR

### Expected Output
- Best checkpoint → `ml/artifacts/checkpoints/resnet_best.pth`
- Training curves, confusion matrix, per-class F1
- Classification report and training summary JSON

In [1]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
if 'notebooks' in str(PROJECT_ROOT):
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/ajitkumarsingh/Desktop/desktop/cattle-breed-classifier-webapp


In [2]:
from ml.src.utils.seed import set_seed
from ml.src.utils.device import get_device
from ml.src.utils.io import load_config
from ml.src.utils.manifests import create_dataloaders
from ml.src.data.transforms import get_train_transforms, get_eval_transforms
from ml.src.models.resnet import CattleResNet
from ml.src.training.trainer import Trainer

set_seed(42)
device = get_device()

InterruptedError: [Errno 4] Interrupted system call

## 1. Load Config & Data

In [ ]:
config = load_config('resnet', PROJECT_ROOT / 'ml' / 'configs')
config['num_classes'] = 26

img_size = config['image']['size']
batch_size = config['training']['batch_size']
arch = config['model']['architecture']

print(f'Backbone: {arch["backbone"]}')
print(f'Pretrained: {arch["pretrained"]}')
print(f'Freeze backbone: {arch["freeze_backbone"]}')
print(f'Unfreeze after: {arch["unfreeze_after_epochs"]} epochs')
print(f'Unfreeze layers: {arch["unfreeze_layers"]}')
print(f'Epochs: {config["training"]["num_epochs"]}')
print(f'LR: {config["training"]["learning_rate"]}  Fine-tune LR: {config["training"]["fine_tune_lr"]}')

In [ ]:
train_transforms = get_train_transforms(img_size=img_size)
eval_transforms = get_eval_transforms(img_size=img_size)

dataloaders = create_dataloaders(
    manifests_dir=PROJECT_ROOT / 'ml' / 'artifacts' / 'manifests',
    data_root=PROJECT_ROOT / 'Cattle_Resized',
    train_transform=train_transforms,
    eval_transform=eval_transforms,
    batch_size=batch_size,
    num_workers=config['data'].get('num_workers', 4),
)

class_names = dataloaders['train'].dataset.classes
print(f'Classes: {len(class_names)}')
print(f'Train: {len(dataloaders["train"].dataset)} | Val: {len(dataloaders["val"].dataset)} | Test: {len(dataloaders["test"].dataset)}')

## 2. Create Model

In [ ]:
model = CattleResNet.from_config(config)

# Count trainable vs frozen params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Total parameters:     {total:,}')
print(f'Trainable (Phase 1):  {trainable:,} ({100*trainable/total:.1f}%)')
print(f'Frozen:               {total - trainable:,}')

## 3. Two-Phase Training

In [ ]:
trainer = Trainer(
    model=model,
    config=config,
    dataloaders=dataloaders,
    class_names=class_names,
    device=device,
    model_name='resnet',
)

# Two-phase training: frozen backbone → partial fine-tuning
training_summary = trainer.train_with_phase_switch(
    phase1_epochs=arch['unfreeze_after_epochs'],
    phase2_lr=config['training']['fine_tune_lr'],
    unfreeze_layers=arch['unfreeze_layers'],
)

## 4. Evaluate on Test Set

In [ ]:
test_results = trainer.evaluate(split='test')

## 5. Save Artifacts

In [ ]:
trainer.save_artifacts()

print('\n=== ResNet50 Transfer Learning Summary ===')
print(f'Best Val Loss:  {training_summary["best_val_loss"]:.4f}')
print(f'Best Val Acc:   {training_summary["best_val_accuracy"]:.4f}')
print(f'Test Accuracy:  {test_results["metrics"]["accuracy"]:.4f}')
print(f'Test Macro F1:  {test_results["metrics"]["macro_f1"]:.4f}')
print(f'Latency:        {test_results["latency"]["avg_ms"]:.2f} ms')
print(f'Model Size:     {test_results["model_size_mb"]:.2f} MB')

## 6. Classification Report

In [ ]:
print(test_results['metrics']['classification_report'])

---
**✅ ResNet50 Transfer Learning complete.** Proceed to `04_vit_transfer_learning.ipynb`.